## CHECKPOINTER

| **Aspect** | **Checkpointer** | **Store** |
|------------|------------------|-----------|
| **Persists** | Graph state snapshots | Application-defined key-value data |
| **Scope** | A single thread | Across threads |
| **Memory type** | Short-term, thread-scoped memory | Long-term, cross-thread memory |
| **Use for** | Conversation continuity, human-in-the-loop, time travel, and fault tolerance | User preferences, facts, and shared knowledge |
| **Access pattern** | Pass a `thread_id` in graph config | Read and write items from nodes or application code |

A checkpointer saves a snapshot of graph state at each super-step, organized into threads. Compile a graph with a checkpointer to enable human-in-the-loop workflows, time travel debugging, fault-tolerant execution, and conversational memory.

## Why Use Checkpointers?

Checkpointers enable several key capabilities in LangGraph:

- **Human-in-the-loop**: Allow users to inspect, interrupt, modify, and resume graph execution.
- **Memory**: Preserve conversation or workflow state across interactions within the same thread.
- **Time travel**: Replay previous executions, debug graph behavior, or branch from earlier checkpoints.
- **Fault tolerance**: Recover from failures by restarting execution from the last successful checkpoint.
- **Pending writes**: Save completed node outputs during partial failures, preventing successful nodes from being re-executed.

In short, checkpointers provide persistence, recovery, memory, and execution control for graph workflows.

## THREADS

A thread is a unique ID or thread identifier assigned to each checkpoint saved by a checkpointer.  
It contains the accumulated state of a sequence of runs. When a run is executed, the state of the underlying graph of the assistant will be persisted to the thread.  
When invoking a graph with a checkpointer, you must specify a thread_id as part of the configurable portion of the config:
  
{"configurable": {"thread_id": "1"}}. 
  
A thread’s current and historical state can be retrieved. To persist state, a thread must be created prior to executing a run.  
The checkpointer uses thread_id as the primary key for storing and retrieving checkpoints.  
Without it, the checkpointer cannot save state or resume execution after an interrupt, since the checkpointer uses thread_id to load the saved state.

# Checkpoint Namespace

Each checkpoint has a `checkpoint_ns` (checkpoint namespace) field that identifies which graph or subgraph it belongs to:

- `""` (empty string): The checkpoint belongs to the parent (root) graph.
- `"node_name:uuid"`: The checkpoint belongs to a subgraph invoked as the given node.
  
For nested subgraphs, namespaces are joined with `|` separators (e.g., `"outer_node:uuid|inner_node:uuid"`).